## 使用@tool装饰器定义工具

### 1. 自定义工具描述: description

举例1:

函数使用 @tool 装饰器修饰以后, 就是一个可以被模型识别的工具了

注意: 在使用 @tool 时, 如果没有提供 description 参数的情况下, 要求函数必须提供 docstring

在同时声明了 description 和 docstring 的情况下, description 的优先级更高

注意: 当我们没有向 @tool 传递 description 参数时, 默认情况下, tool 会将 docstring 整体视为 description
不使用 @tool 装饰器时, docstring 不合法会被视为普通文本, 作为 description, 但如果使用 @tool 时 docstring 不合法, 将会抛出异常

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool

@tool
def get_weather(city: str):
    '''
        使用 @tool 时必须给函数添加 docstring 函数说明
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

例2: 使用 description 参数

注意: 在使用 @tool 时, 如果没有提供 description 参数的情况下, 要求函数必须提供 docstring

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool

@tool(description="获取具体城市的天气情况")
def get_weather(city: str):
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

例3  在同时声明了 description 和 docstring 的情况下, description 的优先级更高

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool

@tool(description="获取具体城市的天气情况")
def get_weather(city: str):
    '''
        使用 @tool 时必须给函数添加 docstring 函数说明
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

例4 

注意: 当我们没有向 @tool 传递 description 参数时, 默认情况下, tool 会将 docstring 整体视为 description
不使用 @tool 装饰器时, docstring 不合法会被视为普通文本, 作为 description, 但如果使用 @tool 时 docstring 不合法, 将会抛出异常

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool

@tool(parse_docstring=True)
def get_weather(city: str):
    '''
    使用 @tool 时必须给函数添加 docstring 函数说明

    Args:
        city: 城市
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

#### 2. 更改工具名称: name_or_callable

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool

@tool(parse_docstring=True, name_or_callable='getWeather')
def get_weather(city: str):
    '''
    使用 @tool 时必须给函数添加 docstring 函数说明

    Args:
        city: 城市
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

### 3. 自定义 args_schema

#### 3.1 使用功能 Pydantic 模型定义

当工具的参数变得复杂, 需要 枚举值, 范围限制 或 更复杂的业务逻辑验证 时, Pydantic 模型时理想的选择, 提供强大的类型检查和数据验证

使用 Pydantic 的主要优势在于能够精确控制工具参数的格式和验证规则, 让大模型更准确的理解如何调用工具

BaseModel 基类
通过继承核心基类 BaseModel 定义数据模型, 从而声明字段结构, 类型约束, 默认值以 0 及校验规则

In [28]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from pydantic import BaseModel

class WeatherInput(BaseModel):
    city: str

@tool(args_schema=WeatherInput)
def get_weather(city: int):
    '''
    使用 @tool 时必须给函数添加 docstring 函数说明
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '使用 @tool 时必须给函数添加 docstring 函数说明',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

例2:

In [29]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    city: str = Field(
        description="具体的城市",
        default="北京"
    )

@tool(args_schema=WeatherInput)
def get_weather(city: int):
    '''
    使用 @tool 时必须给函数添加 docstring 函数说明
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '使用 @tool 时必须给函数添加 docstring 函数说明',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

例3:

In [30]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    city: str
    unit: Literal["celsius", "fahrenheit"]
    include_forecast: bool = Field(
        default=False,
        description="是否包含未来五天的天气预报"
    )

@tool(args_schema=WeatherInput)
def get_weather(city: int, unit: str = 'celsius', include_forecast: bool = False):
    '''
    使用 @tool 时必须给函数添加 docstring 函数说明
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '使用 @tool 时必须给函数添加 docstring 函数说明',
        'parameters': {
            'properties': {
                'city': {'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五天的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['city', 'unit'],
            'type': 'object'
        }
    }
}

### 方式2: 使用 Json Schema 定义

在 LangChain 中, 还可以直接使用 JSON Schema 字典 来定义工具的参数模式, 这种方式提供了极大的灵活性

因为工具参数模式可以基于数据库配置或用户输入在运行时动态生成, 所以这种方式特别适合参数结构需要动态生成的场景

In [31]:
json_schema = {
    'properties': {
        'city': {'type': 'string'},
        'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
        'include_forecast': {
            'default': False,
            'description': '是否包含未来五天的天气预报',
            'type': 'boolean'
        }
    },
    'required': ['city', 'unit'],
    'type': 'object'
}

@tool(args_schema=json_schema)
def get_weather(city: int, unit: str = 'celsius', include_forecast: bool = False):
    '''
    使用 @tool 时必须给函数添加 docstring 函数说明
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '使用 @tool 时必须给函数添加 docstring 函数说明',
        'parameters': {
            'properties': {
                'city': {'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五天的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['city', 'unit'],
            'type': 'object'
        }
    }
}